In [1]:
import os
import tensorflow as tf

os.environ['CUDA_VISIBLE_DEVICES'] = '5'
gpus = tf.config.list_physical_devices('GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
tf.config.experimental.set_virtual_device_configuration(
    gpus[0],
    [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=1024)]
)

I0000 00:00:1775589032.680006 4026350 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import torch

BASE_PATH = "../"
device = "cuda" if torch.cuda.is_available() else "cpu"

HF_TOKEN = ...

In [ ]:
import os
import glob
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample, losses

/data/koposovti/lecture-transcriber/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
all_csv_files = glob.glob(BASE_PATH + "data/linking/*.csv")
df = pd.concat([pd.read_csv(f) for f in all_csv_files], ignore_index=True)
df = df.dropna(subset=['spoken_text'])

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

In [ ]:
def prepare_text_examples(df):
    examples = []
    for _, row in df.iterrows():
        query = "query: " + str(row['spoken_text'])
        
        artifact = "passage: " + str(row['context_content'])
        
        examples.append(InputExample(texts=[query, artifact]))
    return examples


In [6]:
model_name = "intfloat/multilingual-e5-base"
model = SentenceTransformer(model_name, device=device, token=HF_TOKEN)

In [7]:
train_examples = prepare_text_examples(train_df)
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=6)

train_loss = losses.MultipleNegativesRankingLoss(model=model)

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    warmup_steps=100,
    output_path="./models/lecture_linker_e5",
    show_progress_bar=True,
    use_amp=True,
)

Step,Training Loss
500,0.044000
1000,0.041200
1500,0.047600
2000,0.027300
2500,0.031800
3000,0.026600


: 